# LoRA Training Comparison and Analysis

This notebook converts the LoRA training script to Jupyter format with configurable parameters for multiple runs and comparison.

## Features:
- Configurable LoRA training parameters
- Multiple experimental runs with different settings
- Comparative analysis and visualization
- Remote server friendly execution
- Automatic result saving and comparison

---

## 1. Setup and Configuration

In [ ]:
# Define multiple training configurations for comparison
from dataclasses import dataclass, asdict
from typing import List, Dict, Any

@dataclass
class TrainingConfig:
    name: str
    model_name: str = "openai/whisper-small"
    language: str = "zh"
    task: str = "transcribe"
    target_sr: int = 16000
    
    # LoRA parameters
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    target_modules: List[str] = None
    bias: str = "none"
    
    # Training parameters
    train_csv: str = "data/train_original.csv"
    dev_csv: str = "data/dev.csv"
    test_csv: str = "data/test.csv"
    output_dir: str = "models/lora_experiments"
    
    batch_size: int = 16
    learning_rate: float = 1e-4
    num_epochs: int = 3
    warmup_steps: int = 0
    save_steps: int = 500
    eval_steps: int = 500
    logging_steps: int = 100
    
    # Data augmentation
    use_augmentation: bool = False
    augmentation_csv: str = "data/train_augmented.csv"
    
    # Advanced training parameters
    gradient_accumulation_steps: int = 1
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    lr_scheduler_type: str = "constant"
    adam_beta1: float = 0.9
    adam_beta2: float = 0.999
    adam_epsilon: float = 1e-8
    
    # Experiment tracking
    run_id: str = None
    run_name: str = None
    timestamp: str = None

def get_training_configs():
    """Get multiple training configurations for comparison"""
    return [
        TrainingConfig(
            name="baseline_original",
            train_csv="data/train_original.csv",
            lora_r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            batch_size=16,
            learning_rate=1e-4,
            num_epochs=3,
            use_augmentation=False
        ),
        TrainingConfig(
            name="high_rank_augmented",
            train_csv="data/train_augmented.csv",
            lora_r=16,
            lora_alpha=32,
            lora_dropout=0.1,
            batch_size=16,
            learning_rate=1e-4,
            num_epochs=3,
            use_augmentation=False
        ),
        TrainingConfig(
            name="augmented",
            train_csv="data/train_augmented.csv",
            lora_r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            batch_size=16,
            learning_rate=1e-4,
            num_epochs=3,
            use_augmentation=True
        ),
        TrainingConfig(
            name="conservative_augmented",
            train_csv="data/train_augmented.csv",
            lora_r=4,
            lora_alpha=8,
            lora_dropout=0.05,
            batch_size=8,
            learning_rate=5e-5,
            num_epochs=5,
            use_augmentation=False
        ),
        TrainingConfig(
            name="aggressive_augmented",
            train_csv="data/train_augmented.csv",
            lora_r=32,
            lora_alpha=64,
            lora_dropout=0.2,
            batch_size=32,
            learning_rate=2e-4,
            num_epochs=2,
            use_augmentation=False
        ),
        TrainingConfig(
            name="experimental_augmented",
            train_csv="data/train_augmented.csv",
            lora_r=12,
            lora_alpha=24,
            lora_dropout=0.15,
            batch_size=20,
            learning_rate=3e-5,
            num_epochs=4,
            gradient_accumulation_steps=2,
            weight_decay=1e-5,
            max_grad_norm=0.5,
            lr_scheduler_type="cosine",
            adam_beta1=0.95,
            adam_beta2=0.98,
            use_augmentation=False
        ),
    ]

# Get configurations
training_configs = get_training_configs()

print(f"📋 Training Configurations: {len(training_configs)}")
for i, config in enumerate(training_configs):
    print(f"  {i+1}. {config.name}")
    print(f"     LoRA: r={config.lora_r}, alpha={config.lora_alpha}, dropout={config.lora_dropout}")
    print(f"     Training: lr={config.learning_rate}, batch={config.batch_size}, epochs={config.num_epochs}")
    print(f"     Advanced: grad_acc={config.gradient_accumulation_steps}, weight_decay={config.weight_decay}")
    print(f"     Augmentation: {config.use_augmentation}")
    print()


## 1.1 Verify Existing Train / Dev / Test Split

This notebook uses the CSV files you already prepared. It does **not** overwrite `train_original.csv`, `train_augmented.csv`, `dev.csv`, or `test.csv`.


In [ ]:
# Verify existing dataset split only. Do NOT auto-split or overwrite CSV files.

import os
import re
from pathlib import Path
import pandas as pd

TRAIN_ORIGINAL_CSV = "data/train_original.csv"
TRAIN_AUGMENTED_CSV = "data/train_augmented.csv"
DEV_CSV = "data/dev.csv"
TEST_CSV = "data/test.csv"


def standardize_asr_csv(df: pd.DataFrame) -> pd.DataFrame:
    """Make sure CSV uses columns: audio, text."""
    df = df.copy()
    rename_map = {}
    if "audio_path" in df.columns and "audio" not in df.columns:
        rename_map["audio_path"] = "audio"
    if "path" in df.columns and "audio" not in df.columns:
        rename_map["path"] = "audio"
    if "transcript" in df.columns and "text" not in df.columns:
        rename_map["transcript"] = "text"
    if "reference" in df.columns and "text" not in df.columns:
        rename_map["reference"] = "text"
    df = df.rename(columns=rename_map)

    required = {"audio", "text"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}. Current columns: {list(df.columns)}")

    df = df[["audio", "text"]].copy()
    df["audio"] = df["audio"].astype(str).str.strip()
    df["text"] = df["text"].astype(str).str.strip()
    df = df[(df["audio"] != "") & (df["text"] != "")]
    return df


def base_audio_id(path: str) -> str:
    """Return original clip id so augmented versions map back to the same source clip."""
    stem = Path(str(path)).stem
    stem = re.sub(
        r"(_pitch[-+]?\d*|_time[-+]?\d*|_slow|_fast|_stretch|_aug\d*|_noise|_volume|_trimmed|_shortened)$",
        "",
        stem,
    )
    return stem


def load_checked_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    df = standardize_asr_csv(pd.read_csv(path))
    df["base_id"] = df["audio"].map(base_audio_id)
    return df

train_original_df = load_checked_csv(TRAIN_ORIGINAL_CSV)
train_augmented_df = load_checked_csv(TRAIN_AUGMENTED_CSV)
dev_df = load_checked_csv(DEV_CSV)
test_df = load_checked_csv(TEST_CSV)

print("Dataset counts:")
print(f"  train_original.csv:  {len(train_original_df)} rows, {train_original_df['base_id'].nunique()} unique originals")
print(f"  train_augmented.csv: {len(train_augmented_df)} rows, {train_augmented_df['base_id'].nunique()} unique originals")
print(f"  dev.csv:             {len(dev_df)} rows, {dev_df['base_id'].nunique()} unique originals")
print(f"  test.csv:            {len(test_df)} rows, {test_df['base_id'].nunique()} unique originals")

# Check overlap by original clip id, not just exact augmented file path.
sets = {
    "train_original": set(train_original_df["base_id"]),
    "train_augmented": set(train_augmented_df["base_id"]),
    "dev": set(dev_df["base_id"]),
    "test": set(test_df["base_id"]),
}

checks = [
    ("train_original vs dev", sets["train_original"] & sets["dev"]),
    ("train_original vs test", sets["train_original"] & sets["test"]),
    ("train_augmented vs dev", sets["train_augmented"] & sets["dev"]),
    ("train_augmented vs test", sets["train_augmented"] & sets["test"]),
    ("dev vs test", sets["dev"] & sets["test"]),
]

print("\nOverlap check by original clip id:")
for name, overlap in checks:
    print(f"  {name}: {len(overlap)} overlap")
    if overlap:
        print(f"    {sorted(list(overlap))[:10]}")

# Drop helper column before preview.
print("\nPreview train_augmented.csv:")
display(train_augmented_df.drop(columns=["base_id"]).head())
print("\nPreview dev.csv:")
display(dev_df.drop(columns=["base_id"]).head())
print("\nPreview test.csv:")
display(test_df.drop(columns=["base_id"]).head())


## 1.2 Training Imports

Training imports and functions are defined in the next section.


## 2. Training Functions

In [ ]:
# Import necessary libraries for LoRA training
import sys
sys.path.append('scripts')

import os
import json
import time
import re
from datetime import datetime
from dataclasses import asdict
from typing import Any

import torch
import librosa
import numpy as np
from opencc import OpenCC
from datasets import load_dataset
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)
from peft import LoraConfig, get_peft_model
from utils.timing_utils import initialize_timing, log_cell

cc = OpenCC("t2s")


def normalize(text: str) -> str:
    """Normalize Chinese transcript text for training."""
    text = str(text).strip()
    text = cc.convert(text)
    text = re.sub(r"\s+", "", text)
    return text


def load_audio(path: str, target_sr: int = 16000) -> np.ndarray:
    """Load audio as mono 16 kHz float32."""
    audio, _ = librosa.load(path, sr=target_sr, mono=True)
    return audio.astype(np.float32)


def prepare_batch(batch: dict[str, Any], processor: WhisperProcessor, target_sr: int = 16000):
    """Convert one CSV row into Whisper input features and label ids."""
    audio = load_audio(batch["audio"], target_sr=target_sr)

    inputs = processor.feature_extractor(
        audio,
        sampling_rate=target_sr,
        return_tensors="pt",
    )

    text = normalize(batch["text"])
    labels = processor.tokenizer(text).input_ids

    return {
        "input_features": inputs.input_features[0],
        "labels": labels,
    }


class WhisperDataCollator:
    """Pads Whisper input features and labels correctly for Seq2SeqTrainer."""

    def __init__(self, processor: WhisperProcessor):
        self.processor = processor

    def __call__(self, features):
        input_features = [
            {"input_features": f["input_features"]} for f in features
        ]
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt",
        )

        label_features = [
            {"input_ids": f["labels"]} for f in features
        ]
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=True,
            return_tensors="pt",
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100,
        )

        batch["labels"] = labels
        return batch


def train_lora_model(config: TrainingConfig):
    """Train one Whisper LoRA model with the given configuration."""
    print(f"\n🚀 Starting LoRA Training: {config.name}")
    print(f"   LoRA: r={config.lora_r}, alpha={config.lora_alpha}")
    print(f"   Training: lr={config.learning_rate}, batch={config.batch_size}, epochs={config.num_epochs}")
    print(f"   Augmentation config flag: {config.use_augmentation}")
    print(f"   Train CSV: {config.train_csv}")
    print(f"   Dev CSV: {config.dev_csv}")

    if config.run_id is None:
        config.run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
        config.run_name = f"lora_{config.name}_{config.run_id}"
        config.timestamp = datetime.now().isoformat()

    os.makedirs(config.output_dir, exist_ok=True)

    # Load processor and model. Keep processor local so every experiment is self-contained.
    processor = WhisperProcessor.from_pretrained(
        config.model_name,
        language=config.language,
        task=config.task,
    )

    model = WhisperForConditionalGeneration.from_pretrained(config.model_name)
    model.config.use_cache = False

    # Make Whisper training more stable with Trainer.
    model.config.forced_decoder_ids = None
    model.config.suppress_tokens = []

    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.target_modules or ["q_proj", "v_proj"],
        lora_dropout=config.lora_dropout,
        bias=config.bias,
    )

    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Use the pre-split training CSV. The auto-split cell already builds train.csv/dev.csv from train_augmented.csv.
    train_csv = config.train_csv

    dataset = load_dataset(
        "csv",
        data_files={
            "train": train_csv,
            "validation": config.dev_csv,
        },
    )

    dataset = dataset.map(
        lambda batch: prepare_batch(batch, processor, target_sr=config.target_sr),
        remove_columns=dataset["train"].column_names,
    )

    data_collator = WhisperDataCollator(processor)

    training_args = Seq2SeqTrainingArguments(
        output_dir=config.output_dir,
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        warmup_steps=config.warmup_steps,
        num_train_epochs=config.num_epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=config.logging_steps,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        push_to_hub=False,
        report_to=["none"],
        remove_unused_columns=False,
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=torch.cuda.is_available(),
    )

    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        processing_class=processor,
        data_collator=data_collator,
    )

    start_time = time.time()
    trainer.train()
    training_time = time.time() - start_time

    final_output_dir = os.path.join(config.output_dir, config.run_name)
    trainer.save_model(final_output_dir)
    processor.save_pretrained(final_output_dir)

    training_metadata = {
        "run_id": config.run_id,
        "run_name": config.run_name,
        "timestamp": config.timestamp,
        "config": asdict(config),
        "training_time_seconds": training_time,
        "final_model_path": final_output_dir,
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_params": sum(p.numel() for p in model.parameters()),
    }

    metadata_path = os.path.join(final_output_dir, "training_metadata.json")
    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(training_metadata, f, ensure_ascii=False, indent=2)

    print(f"✅ Training completed in {training_time/60:.1f} minutes")
    print(f"📁 Model saved to: {final_output_dir}")
    print(f"📝 Metadata saved to: {metadata_path}")

    return training_metadata


print("🎯 LoRA training functions loaded")




## 3. Run Multiple Training Experiments

In [ ]:
# Initialize timing system
notebook_start_time = time.time()
timer = initialize_timing("results/execution_logs/lora_training_timing.json")

# Select which configurations to run
# Options: run all, run specific, or run single
TRAINING_MODE = "all"  # Options: "all", "single", "specific"

# For single/specific mode, specify which configs
SELECTED_CONFIGS = ["baseline", "high_rank"]  # Only used if mode != "all"

print(f"🎛️  Training Mode: {TRAINING_MODE}")
if TRAINING_MODE == "single":
    print(f"📋 Selected Config: {SELECTED_CONFIGS[0]}")
elif TRAINING_MODE == "specific":
    print(f"📋 Selected Configs: {SELECTED_CONFIGS}")
else:
    print(f"📋 Running All {len(training_configs)} Configurations")

# Determine which configs to run
if TRAINING_MODE == "all":
    configs_to_run = training_configs
elif TRAINING_MODE == "single":
    configs_to_run = [config for config in training_configs if config.name == SELECTED_CONFIGS[0]]
else:  # specific
    configs_to_run = [config for config in training_configs if config.name in SELECTED_CONFIGS]

print(f"\n🚀 Starting {len(configs_to_run)} training experiments...")

# Run training experiments
all_training_results = []

for i, config in enumerate(configs_to_run):
    print(f"\n{'='*60}")
    print(f"Experiment {i+1}/{len(configs_to_run)}: {config.name}")
    
    try:
        with log_cell(f"LoRA Training - {config.name}"):
            result = train_lora_model(config)
            all_training_results.append(result)
    except Exception as e:
        print(f"❌ Training failed for {config.name}: {e}")
        # Add failed result
        failed_result = {
            "run_id": config.run_id or datetime.now().strftime("%Y%m%d_%H%M%S"),
            "run_name": config.run_name or f"lora_{config.name}_failed",
            "timestamp": datetime.now().isoformat(),
            "config": asdict(config),
            "error": str(e),
            "training_time_seconds": 0,
            "final_model_path": None,
        }
        all_training_results.append(failed_result)

print(f"\n🎉 All training experiments completed!")
print(f"📊 Total results: {len(all_training_results)}")

## 4. Training Results Analysis and Visualization

In [ ]:
import os
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use a standard font and remove emoji from plot text to avoid missing glyph warnings.
plt.rcParams["font.family"] = "DejaVu Sans"


def analyze_training_results(results):
    """Analyze and visualize LoRA training results."""
    if not results:
        print("No training results to analyze")
        return None, None

    # Convert successful runs to DataFrame
    df_data = []
    for result in results:
        if "error" not in result:
            df_data.append({
                "run_name": result["run_name"],
                "config_name": result["config"]["name"],
                "lora_r": result["config"]["lora_r"],
                "lora_alpha": result["config"]["lora_alpha"],
                "lora_dropout": result["config"]["lora_dropout"],
                "batch_size": result["config"]["batch_size"],
                "learning_rate": result["config"]["learning_rate"],
                "num_epochs": result["config"]["num_epochs"],
                "use_augmentation": result["config"]["use_augmentation"],
                "training_time_minutes": result["training_time_seconds"] / 60,
                "trainable_params": result["trainable_params"],
                "total_params": result["total_params"],
                "param_efficiency": result["trainable_params"] / result["total_params"] * 100,
                "final_model_path": result["final_model_path"],
            })

    if not df_data:
        print("No successful training results to analyze")
        return None, None

    df = pd.DataFrame(df_data)

    # Create output directory before saving
    os.makedirs("results", exist_ok=True)

    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("LoRA Training Comparison Analysis", fontsize=16, fontweight="bold")

    # 1. Training Time Comparison
    ax1 = axes[0, 0]
    df_sorted = df.sort_values("training_time_minutes")
    ax1.bar(range(len(df_sorted)), df_sorted["training_time_minutes"])
    ax1.set_title("Training Time by Configuration")
    ax1.set_ylabel("Training Time (minutes)")
    ax1.set_xticks(range(len(df_sorted)))
    ax1.set_xticklabels(df_sorted["config_name"], rotation=45, ha="right")
    ax1.grid(True, alpha=0.3)

    for i, v in enumerate(df_sorted["training_time_minutes"]):
        ax1.text(i, v + 0.1, f"{v:.1f}", ha="center", va="bottom")

    # 2. LoRA Parameters Impact
    ax2 = axes[0, 1]
    scatter = ax2.scatter(
        df["lora_r"],
        df["training_time_minutes"],
        s=100,
        alpha=0.7,
        c=df["lora_alpha"],
        cmap="viridis",
    )
    ax2.set_xlabel("LoRA Rank (r)")
    ax2.set_ylabel("Training Time (minutes)")
    ax2.set_title("LoRA Rank vs Training Time")
    ax2.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label("LoRA Alpha")

    # 3. Parameter Efficiency
    ax3 = axes[0, 2]
    ax3.bar(range(len(df)), df["param_efficiency"])
    ax3.set_title("Parameter Efficiency")
    ax3.set_ylabel("Trainable Parameters (%)")
    ax3.set_xticks(range(len(df)))
    ax3.set_xticklabels(df["config_name"], rotation=45, ha="right")
    ax3.grid(True, alpha=0.3)

    for i, v in enumerate(df["param_efficiency"]):
        ax3.text(i, v + 0.02, f"{v:.2f}%", ha="center", va="bottom", fontsize=8)

    # 4. Learning Rate vs Training Time
    ax4 = axes[1, 0]
    scatter = ax4.scatter(
        df["learning_rate"],
        df["training_time_minutes"],
        s=100,
        alpha=0.7,
        c=df["batch_size"],
        cmap="plasma",
    )
    ax4.set_xlabel("Learning Rate")
    ax4.set_ylabel("Training Time (minutes)")
    ax4.set_title("Learning Rate vs Training Time")
    ax4.set_xscale("log")
    ax4.grid(True, alpha=0.3)
    cbar = plt.colorbar(scatter, ax=ax4)
    cbar.set_label("Batch Size")

    # 5. Augmentation Impact
    ax5 = axes[1, 1]
    aug_results = df[df["use_augmentation"] == True]
    no_aug_results = df[df["use_augmentation"] == False]

    labels = []
    values = []
    if len(no_aug_results) > 0:
        labels.append("No Augmentation")
        values.append(no_aug_results["training_time_minutes"].mean())
    if len(aug_results) > 0:
        labels.append("With Augmentation")
        values.append(aug_results["training_time_minutes"].mean())

    ax5.bar(labels, values)
    ax5.set_title("Data Augmentation Impact")
    ax5.set_ylabel("Average Training Time (minutes)")
    ax5.grid(True, alpha=0.3)

    for i, v in enumerate(values):
        ax5.text(i, v + 0.1, f"{v:.1f}", ha="center", va="bottom")

    # 6. Configuration Summary Table
    ax6 = axes[1, 2]
    ax6.axis("off")

    summary_text = "Training Configuration Summary:\n\n"

    fastest = df.loc[df["training_time_minutes"].idxmin()]
    summary_text += f"Fastest: {fastest['config_name']} ({fastest['training_time_minutes']:.1f} min)\n"

    most_efficient = df.loc[df["param_efficiency"].idxmin()]
    summary_text += f"Most Efficient: {most_efficient['config_name']} ({most_efficient['param_efficiency']:.2f}% params)\n"

    highest_rank = df.loc[df["lora_r"].idxmax()]
    summary_text += f"Highest Rank: {highest_rank['config_name']} (r={highest_rank['lora_r']})\n\n"

    if len(aug_results) > 0 and len(no_aug_results) > 0:
        aug_avg = aug_results["training_time_minutes"].mean()
        no_aug_avg = no_aug_results["training_time_minutes"].mean()
        summary_text += f"Augmentation Impact: +{((aug_avg - no_aug_avg) / no_aug_avg * 100):.1f}% time\n"

    summary_text += "\nDetailed Results:\n"
    for _, row in df.iterrows():
        summary_text += (
            f"{row['config_name']}: "
            f"{row['training_time_minutes']:.1f} min, "
            f"{row['param_efficiency']:.2f}% params\n"
        )

    ax6.text(
        0.05,
        0.95,
        summary_text,
        transform=ax6.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
    )

    # Save BEFORE show() to avoid blank output images
    plot_path = f"results/lora_training_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"

    plt.tight_layout()
    plt.savefig(plot_path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Analysis plots saved to: {plot_path}")

    plt.show()
    plt.close(fig)

    return df, plot_path


# Analyze results
if all_training_results:
    analysis_df, analysis_plot = analyze_training_results(all_training_results)
else:
    print("No training results to analyze")


## 5. Export Comprehensive Report

In [ ]:
def export_comprehensive_report(results, analysis_df, plot_path):
    """Export comprehensive training comparison report"""
    if not results or analysis_df is None:
        print("❌ No data to export")
        return
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Create comprehensive report
    report = {
        "experiment_timestamp": datetime.now().isoformat(),
        "experiment_type": "LoRA Training Comparison",
        "total_experiments": len(results),
        "successful_experiments": len([r for r in results if 'error' not in r]),
        "failed_experiments": len([r for r in results if 'error' in r]),
        "training_configurations": [r['config'] for r in results if 'error' not in r],
        "training_results": results,
        "analysis_summary": {
            "fastest_config": analysis_df.loc[analysis_df['training_time_minutes'].idxmin()]['config_name'] if len(analysis_df) > 0 else None,
            "most_efficient_config": analysis_df.loc[analysis_df['param_efficiency'].idxmin()]['config_name'] if len(analysis_df) > 0 else None,
            "highest_rank_config": analysis_df.loc[analysis_df['lora_r'].idxmax()]['config_name'] if len(analysis_df) > 0 else None,
            "avg_training_time": analysis_df['training_time_minutes'].mean() if len(analysis_df) > 0 else 0,
            "augmentation_impact": "Calculated from successful runs"
        },
        "recommendations": {
            "fastest_training": "Use configuration with shortest training time for quick experiments",
            "parameter_efficiency": "Lower LoRA rank and alpha for fewer trainable parameters",
            "augmentation_strategy": "Test with and without data augmentation to see impact",
            "batch_size_optimization": "Larger batch sizes may speed up training but require more memory",
        }
    }
    
    # Save report
    report_path = f"results/lora_training_comparison_report_{timestamp}.json"
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    
    # Save CSV summary
    csv_path = f"results/lora_training_comparison_summary_{timestamp}.csv"
    analysis_df.to_csv(csv_path, index=False)
    
    # Create file list
    file_list_path = f"results/lora_training_files_{timestamp}.txt"
    with open(file_list_path, 'w') as f:
        f.write(f"Analysis Plot: {plot_path}\n")
        f.write(f"Report JSON: {report_path}\n")
        f.write(f"Summary CSV: {csv_path}\n")
        
        for result in results:
            if 'error' not in result and 'final_model_path' in result:
                f.write(f"Model: {result['final_model_path']}\n")
    
    print(f"\n📋 Comprehensive Report Generated:")
    print(f"  📄 JSON Report: {report_path}")
    print(f"  📊 CSV Summary: {csv_path}")
    print(f"  📈 Analysis Plot: {plot_path}")
    print(f"  📋 File List: {file_list_path}")
    
    # Print summary
    successful = [r for r in results if 'error' not in r]
    if successful:
        print(f"\n🎯 Key Findings:")
        print(f"  Successful experiments: {len(successful)}/{len(results)}")
        print(f"  Average training time: {analysis_df['training_time_minutes'].mean():.1f} minutes")
        print(f"  Fastest configuration: {report['analysis_summary']['fastest_config']}")
        print(f"  Most efficient: {report['analysis_summary']['most_efficient_config']}")
    
    return report_path, csv_path, file_list_path

# Export comprehensive report
if all_training_results and 'analysis_df' in locals():
    report_files = export_comprehensive_report(all_training_results, analysis_df, analysis_plot)
else:
    print("❌ Insufficient data for report export")

## 6. Final Completion and Summary

### Training Modes:

#### **All Mode** (Recommended):
```python
TRAINING_MODE = "all"
```
- Runs all 4 predefined configurations
- Best for comprehensive comparison
- Takes longer but provides complete analysis

#### **Single Mode**:
```python
TRAINING_MODE = "single"
SELECTED_CONFIGS = ["baseline"]
```
- Runs only specified configuration
- Good for testing specific parameters
- Faster execution

#### **Specific Mode**:
```python
TRAINING_MODE = "specific"
SELECTED_CONFIGS = ["baseline", "high_rank"]
```
- Runs multiple selected configurations
- Good for targeted experiments

### Configuration Options:

| Config | LoRA Rank | Alpha | Dropout | Batch | LR | Epochs | Augmentation |
|--------|------------|-------|--------|-------|----|--------|--------------|
| baseline_original | 8 | 16 | 0.1 | 16 | 1e-4 | 3 | No |
| high_rank_augmented | 16 | 32 | 0.1 | 16 | 1e-4 | 3 | No |
| augmented | 8 | 16 | 0.1 | 16 | 1e-4 | 3 | Yes |
| conservative_augmented | 4 | 8 | 0.05 | 8 | 5e-5 | 5 | No |
| aggressive_augmented | 32 | 64 | 0.2 | 32 | 2e-4 | 2 | No |

### Expected Outputs:

- **Trained Models**: `models/lora_experiments/lora_{config}_{timestamp}/`
- **Training Metadata**: `training_metadata.json` in each model directory
- **Comparison Plots**: `results/lora_training_comparison_*.png`
- **Analysis Report**: `results/lora_training_comparison_report_*.json`
- **Summary CSV**: `results/lora_training_comparison_summary_*.csv`
- **File List**: `results/lora_training_files_*.txt`

### Usage on Remote Server:

1. **Upload Data**: Ensure `data/train_original.csv`, `data/train_augmented.csv`, `data/dev.csv`, and `data/test.csv` exist
2. **Configure Mode**: Set `TRAINING_MODE` and `SELECTED_CONFIGS` as needed
3. **Run All Cells**: Execute notebook sequentially
4. **Download Results**: All outputs are timestamped for easy download

---

**🎯 Ready for comprehensive LoRA training comparison on remote server!**


In [ ]:
import os
import torch
import pandas as pd
import soundfile as sf
import librosa
from jiwer import cer
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel

import re
from opencc import OpenCC

cc = OpenCC("t2s")


def normalize_chinese_text(text):
    """
    Normalize Chinese ASR output.
    """

    if text is None:
        return ""

    # Convert traditional -> simplified
    text = cc.convert(text)

    # Remove spaces
    text = text.replace(" ", "")

    # Keep ONLY Chinese characters
    text = re.sub(r"[^\u4e00-\u9fff]", "", text)

    return text

def remove_repeated_phrases(text):
    words = text.split()

    cleaned = []
    prev = None
    repeat_count = 0

    for w in words:
        if w == prev:
            repeat_count += 1

            # allow at most 2 repeats
            if repeat_count < 2:
                cleaned.append(w)
        else:
            repeat_count = 0
            cleaned.append(w)

        prev = w

    return " ".join(cleaned)

def evaluate_lora_cer(
    base_model_name,
    lora_model_path,
    eval_csv,
    output_csv,
    target_sr=16000,
    language="Chinese",
    task="transcribe"
):
    print("=" * 60)
    print("Starting CER Evaluation")
    print("=" * 60)

    print(f"Base model: {base_model_name}")
    print(f"LoRA model: {lora_model_path}")
    print(f"Evaluation CSV: {eval_csv}")

    processor = WhisperProcessor.from_pretrained(
        base_model_name,
        language=language,
        task=task
    )

    print("Loading base Whisper model...")
    base_model = WhisperForConditionalGeneration.from_pretrained(base_model_name)

    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(base_model, lora_model_path)

    model.eval()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    print(f"Using device: {device}")

    df = pd.read_csv(eval_csv)

    print(f"Loaded {len(df)} evaluation samples")

    predictions = []
    references = []
    audio_paths = []

    print("\nRunning inference...")
    print("-" * 60)

    for idx, row in df.iterrows():
        audio_path = row["audio"]
        reference = row["text"]

        print(f"\n[{idx+1}/{len(df)}] Processing:")
        print(f"Audio: {os.path.basename(audio_path)}")

        y, sr = sf.read(audio_path)

        if len(y.shape) > 1:
            y = y.mean(axis=1)

        if sr != target_sr:
            y = librosa.resample(y, orig_sr=sr, target_sr=target_sr)

        duration = len(y) / target_sr
        print(f"Duration: {duration:.2f}s")

        inputs = processor(
            y,
            sampling_rate=target_sr,
            return_tensors="pt"
        )

        input_features = inputs.input_features.to(device)

        with torch.no_grad():
            predicted_ids = model.generate(
                input_features,
                language="zh",
                task="transcribe",
                temperature=0.0,
                no_repeat_ngram_size=3,
                repetition_penalty=1.2,
                length_penalty=1.0,
            )

        prediction = processor.batch_decode(
            predicted_ids,
            skip_special_tokens=True
        )[0]

        prediction = normalize_chinese_text(prediction)
        reference = normalize_chinese_text(reference)

        # Repetition Detection (Optional but Recommended)
        # prediction = remove_repeated_phrases(prediction)

        print(f"Reference : {reference}")
        print(f"Prediction: {prediction}")

        sample_cer = cer([reference], [prediction])
        print(f"Sample CER: {sample_cer:.4f}")

        audio_paths.append(audio_path)
        references.append(reference)
        predictions.append(prediction)

    print("\n" + "=" * 60)
    print("Final CER Evaluation")
    print("=" * 60)

    overall_cer = cer(references, predictions)

    print(f"Overall CER: {overall_cer:.4f}")

    results_df = pd.DataFrame({
        "audio": audio_paths,
        "reference": references,
        "prediction": predictions
    })

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)

    results_df.to_csv(
        output_csv,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Saved predictions to: {output_csv}")

    return overall_cer, results_df

In [ ]:
# Evaluate the exact LoRA models trained in this notebook
# This avoids accidentally evaluating old hardcoded adapter folders.

base_model_name = "openai/whisper-small"
eval_csv = "data/test.csv"

if "all_training_results" not in globals() or not all_training_results:
    raise RuntimeError("Run the training cell first so all_training_results contains the current model paths.")

cer_results = []
evaluation_outputs = {}

for result in all_training_results:
    if result.get("error") or not result.get("final_model_path"):
        print(f"Skipping failed run: {result.get('run_name')}")
        continue

    config_name = result["config"]["name"]
    model_path = result["final_model_path"]
    output_csv = f"results/predictions/{config_name}_predictions.csv"

    print("\n" + "#" * 80)
    print(f"Evaluating CURRENT run: {config_name}")
    print(f"Adapter path: {model_path}")

    model_cer, model_results = evaluate_lora_cer(
        base_model_name=base_model_name,
        lora_model_path=model_path,
        eval_csv=eval_csv,
        output_csv=output_csv,
    )

    cer_results.append({"model": f"LoRA {config_name}", "cer": model_cer, "path": model_path})
    evaluation_outputs[config_name] = model_results

cer_df = pd.DataFrame(cer_results).sort_values("cer")
display(cer_df)


In [ ]:
# Diagnostic: check whether different adapters produce identical predictions
# If predictions are identical, the adapter may not be affecting generation, or training did not change weights enough.

if "evaluation_outputs" not in globals() or len(evaluation_outputs) < 2:
    print("Run the dynamic evaluation cell above first.")
else:
    names = list(evaluation_outputs.keys())
    ref_name = names[0]
    ref_pred = evaluation_outputs[ref_name]["prediction"].tolist()

    for name in names[1:]:
        pred = evaluation_outputs[name]["prediction"].tolist()
        same = sum(a == b for a, b in zip(ref_pred, pred))
        total = min(len(ref_pred), len(pred))
        print(f"{ref_name} vs {name}: identical predictions {same}/{total}")

    print("\nIf all are 18/18 identical, check:")
    print("1. You are evaluating newly trained final_model_path, not old hardcoded paths.")
    print("2. warmup_steps is not larger than total training steps.")
    print("3. Training loss actually decreases.")
    print("4. Baseline and augmented are using different source CSVs if you want to compare augmentation.")


In [ ]:
# cer_results is now produced by the dynamic evaluation cell above.
# This fallback keeps the plot cell working if you manually evaluated models earlier.

if "cer_results" not in globals() or not cer_results:
    cer_results = [
        {"model": "LoRA Baseline", "cer": baseline_cer},
        {"model": "LoRA High Rank", "cer": high_rank_cer},
        {"model": "LoRA Augmented", "cer": augmented_cer},
        {"model": "LoRA Aggressive", "cer": aggressive_cer},
    ]


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
from datetime import datetime

cer_df = pd.DataFrame(cer_results)
cer_df = cer_df.sort_values("cer")

plt.figure(figsize=(10, 6))
bars = plt.bar(cer_df["model"], cer_df["cer"])

plt.title("CER Comparison Across LoRA Models", fontsize=14, fontweight="bold")
plt.xlabel("Model Configuration")
plt.ylabel("Character Error Rate (CER)")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)

for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        height,
        f"{height:.4f}",
        ha="center",
        va="bottom"
    )

best_model = cer_df.iloc[0]
plt.figtext(
    0.5,
    -0.05,
    f"Best model: {best_model['model']} with CER = {best_model['cer']:.4f}",
    ha="center",
    fontsize=11
)

os.makedirs("results/figures", exist_ok=True)
plot_path = f"results/figures/cer_comparison_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"

plt.tight_layout()
plt.savefig(plot_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

print(f"Saved CER plot to: {plot_path}")
cer_df